In [ ]:
from pathlib import Path
import time

import geopandas as gpd
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


# ------------------------------------------------------------
# Change this later
# ------------------------------------------------------------
POINT_SHP = Path(r"C:\path\to\your\five_points.shp")

OUT_CSV = POINT_SHP.with_name(POINT_SHP.stem + "_coordinates_villages.csv")


# ------------------------------------------------------------
# Read point shapefile
# ------------------------------------------------------------
gdf = gpd.read_file(POINT_SHP)

if gdf.empty:
    raise ValueError("The shapefile is empty.")

if len(gdf) != 5:
    print(f"Warning: expected 5 points, but found {len(gdf)} features.")

if gdf.crs is None:
    raise ValueError(
        "The shapefile has no CRS. Define the correct CRS first, otherwise coordinates will be wrong."
    )

# Keep only point geometries
gdf = gdf[gdf.geometry.type == "Point"].copy()

if gdf.empty:
    raise ValueError("No point geometries found in the shapefile.")

# Convert to WGS84 lon/lat
gdf_wgs84 = gdf.to_crs(epsg=4326).copy()

gdf_wgs84["longitude"] = gdf_wgs84.geometry.x
gdf_wgs84["latitude"] = gdf_wgs84.geometry.y


# ------------------------------------------------------------
# Reverse geocode village/place names
# ------------------------------------------------------------
geolocator = Nominatim(
    user_agent="master_thesis_point_reverse_geocoder"
)

reverse = RateLimiter(
    geolocator.reverse,
    min_delay_seconds=1,
    max_retries=2,
    error_wait_seconds=2,
    swallow_exceptions=True,
)

def get_place_name(lat, lon):
    location = reverse(
        (lat, lon),
        language="en",
        addressdetails=True,
        zoom=14,
    )

    if location is None:
        return None

    address = location.raw.get("address", {})

    # Try increasingly broader settlement labels
    for key in [
        "village",
        "town",
        "city",
        "municipality",
        "county",
        "state_district",
        "state",
        "country",
    ]:
        if key in address:
            return address[key]

    return location.address


place_names = []

for _, row in gdf_wgs84.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    place_names.append(get_place_name(lat, lon))

gdf_wgs84["village_name"] = place_names


# ------------------------------------------------------------
# Export table
# ------------------------------------------------------------
result = pd.DataFrame({
    "id": range(1, len(gdf_wgs84) + 1),
    "longitude": gdf_wgs84["longitude"],
    "latitude": gdf_wgs84["latitude"],
    "village_name": gdf_wgs84["village_name"],
})

result.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(result)
print(f"\nSaved to: {OUT_CSV}")

In [5]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyogrio
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


# ------------------------------------------------------------
# Input File Geodatabase and layer
# ------------------------------------------------------------
GDB_PATH = Path(r"C:\Users\franz\Documents\ArcGIS\Projects\GeoAI_TestData\GeoAI_TestData.gdb")
LAYER_NAME = "Improved_Locations"

OUT_CSV = GDB_PATH.parent / f"{LAYER_NAME}_coordinates_villages.csv"


# ------------------------------------------------------------
# Check available layers
# ------------------------------------------------------------
print("Available layers in geodatabase:")
layers = pyogrio.list_layers(GDB_PATH)

for layer in layers:
    print(layer)

available_layer_names = [layer[0] for layer in layers]

if LAYER_NAME not in available_layer_names:
    raise ValueError(
        f"Layer '{LAYER_NAME}' not found in {GDB_PATH}.\n"
        f"Available layers are: {available_layer_names}"
    )


# ------------------------------------------------------------
# Read point feature class from FileGDB
# ------------------------------------------------------------
gdf = gpd.read_file(GDB_PATH, layer=LAYER_NAME)

if gdf.empty:
    raise ValueError("The feature class is empty.")

if len(gdf) != 5:
    print(f"Warning: expected 5 points, but found {len(gdf)} features.")

if gdf.crs is None:
    raise ValueError(
        "The feature class has no CRS. Define the correct CRS first, otherwise coordinates will be wrong."
    )


# ------------------------------------------------------------
# Keep only point geometries
# ------------------------------------------------------------
gdf = gdf[gdf.geometry.type == "Point"].copy()

if gdf.empty:
    raise ValueError("No point geometries found in the feature class.")


# ------------------------------------------------------------
# Convert to WGS84 lon/lat
# ------------------------------------------------------------
gdf_wgs84 = gdf.to_crs(epsg=4326).copy()

gdf_wgs84["longitude"] = gdf_wgs84.geometry.x
gdf_wgs84["latitude"] = gdf_wgs84.geometry.y


# ------------------------------------------------------------
# Reverse geocode village/place names
# ------------------------------------------------------------
geolocator = Nominatim(
    user_agent="master_thesis_point_reverse_geocoder"
)

reverse = RateLimiter(
    geolocator.reverse,
    min_delay_seconds=1,
    max_retries=2,
    error_wait_seconds=2,
    swallow_exceptions=True,
)


def get_place_name(lat, lon):
    location = reverse(
        (lat, lon),
        language="en",
        addressdetails=True,
        zoom=14,
    )

    if location is None:
        return None

    address = location.raw.get("address", {})

    for key in [
        "village",
        "hamlet",
        "town",
        "city",
        "municipality",
        "county",
        "state_district",
        "state",
        "country",
    ]:
        if key in address:
            return address[key]

    return location.address


place_names = []

for _, row in gdf_wgs84.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    place_names.append(get_place_name(lat, lon))

gdf_wgs84["village_name"] = place_names


# ------------------------------------------------------------
# Export table
# ------------------------------------------------------------
result = pd.DataFrame(
    {
        "id": range(1, len(gdf_wgs84) + 1),
        "longitude": gdf_wgs84["longitude"].values,
        "latitude": gdf_wgs84["latitude"].values,
        "village_name": gdf_wgs84["village_name"].values,
    }
)

result.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(result)
print(f"\nSaved to: {OUT_CSV}")

Available layers in geodatabase:
['aoi' 'MultiPolygon Z']
['export_JSONToFeatures' 'MultiPolygon Z']
['export_JSONToFeatures_ExportFeatures' 'MultiPolygon Z']
['Improved_Locations' 'Point Z']
   id  longitude   latitude         village_name
0   1  40.708186 -14.568532               Nacala
1   2  85.521601  27.640378               Banepa
2   3   1.960051  13.584270           Saga Fondo
3   4 -10.792940   6.338041           Sayon Town
4   5 -93.757355  16.803258  Francisco I. Madero

Saved to: C:\Users\franz\Documents\ArcGIS\Projects\GeoAI_TestData\Improved_Locations_coordinates_villages.csv
